# 07-after-konst-field

Preprocessing initial conditions

**Goal:** Get the trj of the initial conditions for the simulation.

The relevant trj are the fast (regular) and the very slow simulations at 5mT.

Then export that frame into the data directory as initial conditions for the `07-after-konst-field` script to simulate.

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import subprocess
from pathlib import Path

sys.path.insert(0, "../../icenumerics/")
sys.path.insert(0, "../auxnumerics/")
sys.path.insert(0, "../")  # for parameters.py


import icenumerics as ice
import concurrent.futures
import auxiliary as aux
import vertices as vrt

from parameters import params
from tqdm import tqdm
import importlib
import argparse

ureg = ice.ureg
idx = pd.IndexSlice

In [2]:
DISKS = {
    'aura':'4051FBFC5BE73E7F',
}

In [3]:
def get_mountpoint(uuid, fallback=None):
    """
        Gets the mountpoint of the mounted disk with specified UUID.
        
        Parameters:
        ----------
        uuid: str - Disk UUID
        fallback: any - (optional) Specified fallback option or function.
    """
    try:
        mountpoint = subprocess.check_output(
            ["findmnt", "-rn", "-S", f"UUID={uuid}", "-o", "TARGET"],
            text=True
        ).strip()
        return (mountpoint,False) if mountpoint else fallback
    except subprocess.CalledProcessError:
        return fallback

def get_repo_root(as_fallback=False):
    """
        Returns the root of the current git repository.
        Runs `git rev-parse --show-toplevel`
        
        Parameters:
        ----------
        as_fallback: bool - Flag that specifies if this comes from a fallback
    """
    repopath = subprocess.check_output(
        ["git", "rev-parse", "--show-toplevel"], text=True
    ).strip()

    if as_fallback:
        return repopath, as_fallback
    else:
        return repopath

In [8]:
# deciding on a data root
DATA_ROOT, as_fallback = get_mountpoint(
    DISKS['aura'], 
    fallback=get_repo_root(as_fallback=True)
)

if as_fallback: # data is stored locally
    DATA_ROOT = os.path.join(DATA_ROOT, 'data')
else: # data is in external disk
    DATA_ROOT = os.path.join(DATA_ROOT, 'BIG', 'stuckgs', 'data')

DATA_ROOT

'/run/media/holo/aura/BIG/stuckgs/data'

In [9]:
# setting up the rest of the datadir
SIZE = 30

In [10]:
def save_frame(params, src_file, realization, B=5, type_sim='fast'):
    trj = ice.trajectory(src_file)
    trj.load()
    
    # esta bien menso como hice esto, pero no recuerdo los fps
    time_array = trj.trj['t'].values
    field = params["ramp_rate"] * time_array * (time_array<=params["konst_time"])
    field = field + params["max_field"] * (time_array>params["konst_time"])
    
    time_key = time_array[field==B][0]
    
    initial_condition = trj.trj[
        trj.trj['t'] == time_key
    ].reset_index().drop(columns=["frame"])
    
    target_dir = os.path.join(
        get_repo_root(),
        'data',
        '07-after-konst-field',
        'initial-conditions'
    )

    tg_file = os.path.join(target_dir, f'{type_sim}-{realization}.csv')
    initial_condition.to_csv(tg_file, index=False)

In [14]:
fast_params = {
    # the ramp rante mT/s
    "ramp_rate": 10/300,
    # the actual max field
    "max_field": 10,
    # time after which the ramp stopped
    "konst_time": 300 
}

In [15]:
project = 'sims'
data_dir = os.path.join(DATA_ROOT, project, str(SIZE))

print(f'Data is read from: {data_dir}')

for realization in range(1,11):
    src_file = os.path.join(data_dir,'ctrj',f'xtrj{realization}.csv')
    print(f'{src_file = }')
    
    save_frame(
        fast_params, 
        src_file,
        realization, 
        B=5, 
        type_sim='fast'
    )

Data is read from: /run/media/holo/aura/BIG/stuckgs/data/sims/30
src_file = '/run/media/holo/aura/BIG/stuckgs/data/sims/30/ctrj/xtrj1.csv'
src_file = '/run/media/holo/aura/BIG/stuckgs/data/sims/30/ctrj/xtrj2.csv'
src_file = '/run/media/holo/aura/BIG/stuckgs/data/sims/30/ctrj/xtrj3.csv'
src_file = '/run/media/holo/aura/BIG/stuckgs/data/sims/30/ctrj/xtrj4.csv'
src_file = '/run/media/holo/aura/BIG/stuckgs/data/sims/30/ctrj/xtrj5.csv'
src_file = '/run/media/holo/aura/BIG/stuckgs/data/sims/30/ctrj/xtrj6.csv'
src_file = '/run/media/holo/aura/BIG/stuckgs/data/sims/30/ctrj/xtrj7.csv'
src_file = '/run/media/holo/aura/BIG/stuckgs/data/sims/30/ctrj/xtrj8.csv'
src_file = '/run/media/holo/aura/BIG/stuckgs/data/sims/30/ctrj/xtrj9.csv'
src_file = '/run/media/holo/aura/BIG/stuckgs/data/sims/30/ctrj/xtrj10.csv'


In [16]:
slow_params = {
    # the ramp rante mT/s
    "ramp_rate": 5/15_000,
    # the actual max field
    "max_field": 5,
    # time after which the ramp stopped
    "konst_time": 15_000 
}

In [17]:
project = 'sims_superslow_short'
data_dir = os.path.join(DATA_ROOT, project)

print(f'Data is read from: {data_dir}')

for realization in range(1,11):
    src_file = os.path.join(data_dir,f'trj{realization}.csv')
    print(f'{src_file = }')
    
    save_frame(
        slow_params, 
        src_file,
        realization, 
        B=5, 
        type_sim='slow'
    )

Data is read from: /run/media/holo/aura/BIG/stuckgs/data/sims_superslow_short
src_file = '/run/media/holo/aura/BIG/stuckgs/data/sims_superslow_short/trj1.csv'
src_file = '/run/media/holo/aura/BIG/stuckgs/data/sims_superslow_short/trj2.csv'
src_file = '/run/media/holo/aura/BIG/stuckgs/data/sims_superslow_short/trj3.csv'
src_file = '/run/media/holo/aura/BIG/stuckgs/data/sims_superslow_short/trj4.csv'
src_file = '/run/media/holo/aura/BIG/stuckgs/data/sims_superslow_short/trj5.csv'
src_file = '/run/media/holo/aura/BIG/stuckgs/data/sims_superslow_short/trj6.csv'
src_file = '/run/media/holo/aura/BIG/stuckgs/data/sims_superslow_short/trj7.csv'
src_file = '/run/media/holo/aura/BIG/stuckgs/data/sims_superslow_short/trj8.csv'
src_file = '/run/media/holo/aura/BIG/stuckgs/data/sims_superslow_short/trj9.csv'
src_file = '/run/media/holo/aura/BIG/stuckgs/data/sims_superslow_short/trj10.csv'
